# LLM basics: what really happens inside a language model

We'll open up a real language model and look at what it does step by step:

1. How text becomes **tokens**
2. How the model predicts the **next token**
3. How text is **generated** one token at a time – and what **temperature** does
4. The difference between a **base** model and a **chat** model

We use two small models so everything runs on a laptop CPU:
- [GPT-2](https://huggingface.co/openai-community/gpt2) (OpenAI, 2019, 124M parameters, ~550 MB) – a classic *base* model,
- [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) (Alibaba, 2024, 0.5B parameters, ~1 GB) – a small modern *chat* model.

Today's frontier models work on the same principles – they're just ~1000× bigger and trained on far more data.

## Setup

We use the [Hugging Face `transformers`](https://huggingface.co/docs/transformers) library with PyTorch (Class 5). The models download automatically the first time (then they're cached on your disk).

Warnings about `HF_TOKEN` ("unauthenticated requests") or about symlinks on Windows are harmless – you can ignore them.

In [ ]:
%pip install -q transformers torch

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)
model.eval()   # inference mode: no dropout

print(f"{MODEL}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters, "
      f"vocabulary of {tokenizer.vocab_size} tokens")

## 1. Tokens

The model never sees letters or words – only **token IDs** from its vocabulary (subword tokenization, Class 3). GPT-2 uses Byte Pair Encoding with ~50 000 tokens.

In [ ]:
def show_tokens(text):
    ids = tokenizer(text)["input_ids"]
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{len(ids):>2} tokens: {pieces}")


show_tokens("I love deep learning.")
show_tokens("Tokenization is surprisingly important.")
show_tokens("Uwielbiam głębokie uczenie.")
show_tokens("strawberry")

Look at the Polish sentence: GPT-2 was trained mostly on English, so Polish words are cut into many small pieces (sometimes even single bytes, shown as `�`). More tokens = more expensive, slower and a smaller effective context. Modern models have bigger, multilingual vocabularies – but this is still why the same text in Polish usually costs more tokens than in English.

And `strawberry`? It's only a couple of tokens – the model never "sees" the three r's. That's the famous reason LLMs used to fail at counting letters.

## 2. Predicting the next token

This is **the only thing** a language model does: given a sequence of tokens, output a probability for every token in the vocabulary to be the next one. That's a softmax over ~50 000 classes – multi-class classification, exactly like Reuters in Class 2, just much bigger.

In [ ]:
@torch.no_grad()   # we're not training, so no gradients needed
def next_token_probs(text):
    inputs = tokenizer(text, return_tensors="pt")
    logits = model(**inputs).logits[0, -1]   # scores for the token after the last one
    return logits, torch.softmax(logits, dim=-1)


def show_top(text, k=10):
    _, probs = next_token_probs(text)
    top = torch.topk(probs, k)
    print(f"{text!r} →")
    for p, i in zip(top.values, top.indices):
        print(f"   {tokenizer.decode([int(i)])!r:>14}  {p:.3f}")


show_top("The capital of France is")

In [ ]:
show_top("I went to the kitchen to make a cup of")

## 3. Generating text – one token at a time

Generation is a loop: predict the next token → pick one → append it → repeat. Here it is written by hand, so you can see there's no magic.

How do we **pick** the next token?
- **Greedy** (`temperature=0`): always take the most likely one. Safe, but boring and often repetitive.
- **Sampling with temperature**: draw randomly from the distribution after dividing the scores (logits) by the temperature:
  - temperature < 1 → the distribution gets *sharper* (more predictable),
  - temperature > 1 → it gets *flatter* (more surprising, eventually nonsense).

Good text sits somewhere between predictability and surprise (the "sampling strategy" part of the text-generation chapter in Chollet's book explains this nicely).

In [ ]:
@torch.no_grad()
def generate(prompt, n_tokens=30, temperature=0.0, seed=0):
    torch.manual_seed(seed)                                   # reproducible sampling
    ids = tokenizer(prompt, return_tensors="pt")["input_ids"]
    for _ in range(n_tokens):
        logits = model(input_ids=ids).logits[0, -1]
        if temperature == 0:
            next_id = torch.argmax(logits)                    # greedy
        else:
            probs = torch.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)[0]   # sample
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)     # append and repeat
    return tokenizer.decode(ids[0])


prompt = "In the future, artificial intelligence will"
for t in [0.0, 0.7, 1.0, 1.5]:
    print(f"--- temperature {t}")
    print(generate(prompt, temperature=t), "\n")

What temperature does to the probabilities themselves:

In [ ]:
logits, _ = next_token_probs("My favourite food is")
top_ids = torch.topk(logits, 10).indices
labels = [tokenizer.decode([int(i)]) for i in top_ids]

plt.figure(figsize=(12, 3))
for n, t in enumerate([0.5, 1.0, 2.0]):
    probs = torch.softmax(logits / t, dim=-1)[top_ids]
    plt.subplot(1, 3, n + 1)
    plt.bar(labels, probs.numpy())
    plt.title(f"temperature {t}")
    plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

In practice you don't write the loop yourself – `generate()` does it (much faster: it caches the attention computations of previous tokens, so each step only processes the new one). `top_p` is another common trick: sample only from the smallest set of tokens that covers e.g. 90% of the probability.

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt")
out = model.generate(**inputs, max_new_tokens=40, do_sample=True, temperature=0.8, top_p=0.9,
                     pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## 4. Base model vs. chat model

GPT-2 is a **base model**: it continues text. Ask it a question and it doesn't really *answer* – it just continues the document, often by repeating itself or drifting off into something that looks like a forum thread.

In [ ]:
print(generate("What is the capital of France?", n_tokens=30))

A **chat (instruct) model** went through post-training on conversations (Class 6 notes). It also expects a special **chat template** – the conversation is turned into one long text with special tokens marking who says what. Let's look at it.

In [ ]:
CHAT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL)
chat_model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL)
chat_model.eval()

messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer briefly."},
    {"role": "user", "content": "What is the capital of France?"},
]
text = chat_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(text)

That's all a "conversation" is for the model: one text, and the model predicts what comes after `<|im_start|>assistant`. Now let it answer:

In [ ]:
def chat(messages, max_new_tokens=100):
    text = chat_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = chat_tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        out = chat_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[0, inputs["input_ids"].shape[1]:]          # only the answer, not the prompt
    return chat_tokenizer.decode(new_tokens, skip_special_tokens=True)


print(chat(messages))

In [ ]:
print(chat([
    {"role": "system", "content": "You are a patient teacher of deep learning."},
    {"role": "user", "content": "Explain in two sentences what a loss function is."},
]))

A 0.5B model is tiny by today's standards – it will make mistakes and invent facts. Try asking it something tricky and see where it breaks. The big models work exactly the same way, they are just much better at it.

## Your turn

1. Find a prompt where GPT-2's greedy output starts repeating itself. Does a temperature of 0.7 fix it?
2. Compare how many tokens the same paragraph takes in English and in Polish – for GPT-2 and for the Qwen tokenizer (`chat_tokenizer`). Which one is more "Polish-friendly"?
3. Change the system prompt of the chat model (e.g. "Answer like a pirate", "Answer only with yes or no"). How strictly does it follow it?
4. Ask the chat model a question about something that happened recently. What does it say – and why?